# Module 1–2 — Create TWO Direct Lake semantic models (one per line of business)

Builds **two** Direct Lake semantic models over the **gold** tables, one for each line of business, then enriches each with relationships, KPI measures, hierarchies, and AI-readiness metadata using **semantic-link-labs**:

| Model | Line of business | Tables |
|---|---|---|
| `PaymentsDepositsModel` | Payments & Deposits | dim_customer, dim_account, dim_merchant, dim_product, dim_branch, dim_date, fact_transaction, fact_fraud_alert |
| `LendingCreditRiskModel` | Lending & Credit Risk | dim_customer *(shared)*, fact_loan, fact_delinquency |

Both models share the entity-resolved **`dim_customer`** (`customerKey`). Each model answers questions **within** its line of business.

**The cross-domain question** — *"delinquent borrowers who also had fraud alerts" = 13* — spans both models, so **no single semantic model can answer it**. That is exactly the job of the **ontology** (Module 3–4): it spans both subject areas and unifies them through the shared `Customer`. This notebook builds the two BI models; the ontology notebook builds the cross-domain layer on top.

The cells run **top to bottom in one pass** and are idempotent (`overwrite=True`).

**Before running:**
1. Run `nb_load_files_to_tables` first so the `gold.*` tables exist.
2. Attach this notebook to the **same lakehouse** (left **Explorer** → **Add**).
3. Set `LAKEHOUSE` in the config cell to your lakehouse name (here: `FabricIQ`).
4. **Run all** (▶▶, not cell-by-cell). The first cell installs the library; the kernel may take a minute.

**Success looks like:** the final cell prints each model's KPIs (Payments: 48,710 txns, $126.8M deposits · Lending: $58.8M loan book, 15.5% delinquency).


In [ ]:
# Install the Fabric semantic-model toolkit (semantic-link-labs).
%pip install semantic-link-labs


In [ ]:
import sempy_labs as labs

LAKEHOUSE = "FabricIQ"   # <-- your lakehouse name (holds the gold tables)
SCHEMA = "gold"

# ---- Model A: Payments & Deposits ----
A_TABLES = ["dim_customer", "dim_account", "dim_merchant", "dim_product", "dim_branch",
            "dim_date", "fact_transaction", "fact_fraud_alert"]
A_RELS = [
    ("fact_transaction", "customerKey", "dim_customer", "customerKey"),
    ("fact_transaction", "merchantId", "dim_merchant", "merchantId"),
    ("fact_transaction", "accountId", "dim_account", "accountId"),
    ("fact_transaction", "dateKey", "dim_date", "dateKey"),
    ("fact_fraud_alert", "customerKey", "dim_customer", "customerKey"),
    ("dim_account", "productId", "dim_product", "productId"),
    ("dim_account", "branchId", "dim_branch", "branchId"),
]
A_MEASURES = [
    ("fact_transaction", "Transaction Count", "COUNTROWS(fact_transaction)", "Volumes", "Number of card transactions."),
    ("fact_transaction", "Transaction Amount", "SUM(fact_transaction[amount])", "Volumes", "Total transaction spend in USD."),
    ("fact_transaction", "Fee Income", "SUM(fact_transaction[amount]) * 0.015", "Volumes", "Estimated fee income at 1.5% of transaction amount."),
    ("fact_transaction", "Fraud Txn Count", "CALCULATE(COUNTROWS(fact_transaction), fact_transaction[isFraud] = TRUE())", "Fraud", "Count of confirmed-fraud transactions."),
    ("fact_transaction", "Fraud Loss Amount", "CALCULATE(SUM(fact_transaction[amount]), fact_transaction[isFraud] = TRUE())", "Fraud", "USD amount of confirmed-fraud transactions."),
    ("fact_transaction", "Fraud Loss Rate", "DIVIDE([Fraud Loss Amount], [Transaction Amount])", "Fraud", "Fraud Loss Amount / Transaction Amount. A lower rate is better."),
    ("dim_account", "Deposit Balance", "SUM(dim_account[balance])", "Balances", "Total deposit balances across accounts in USD."),
    ("fact_fraud_alert", "Open Fraud Alerts", 'CALCULATE(COUNTROWS(fact_fraud_alert), fact_fraud_alert[status] = "Open")', "Fraud", "Count of fraud alerts currently in Open status."),
    ("fact_fraud_alert", "Customers With Fraud Alert", "DISTINCTCOUNT(fact_fraud_alert[customerKey])", "Fraud", "Distinct customers with at least one fraud alert."),
]
A_HIERS = {
    "dim_date": ("Calendar", ["year", "quarter", "month"]),
    "dim_branch": ("Geography", ["region", "name"]),
    "dim_product": ("Product Hierarchy", ["family", "name"]),
}

# ---- Model B: Lending & Credit Risk ----
B_TABLES = ["dim_customer", "fact_loan", "fact_delinquency"]
B_RELS = [
    ("fact_loan", "customerKey", "dim_customer", "customerKey"),
    ("fact_delinquency", "customerKey", "dim_customer", "customerKey"),
]
B_MEASURES = [
    ("fact_loan", "Loan Book Balance", "SUM(fact_loan[balance])", "Lending", "Total outstanding loan balance in USD."),
    ("fact_loan", "Loan Count", "COUNTROWS(fact_loan)", "Lending", "Number of loans."),
    ("fact_loan", "Avg Loan Rate", "AVERAGE(fact_loan[rate])", "Lending", "Average loan interest rate percent."),
    ("fact_loan", "Delinquency Rate", "DIVIDE(CALCULATE(COUNTROWS(fact_loan), fact_loan[dpd] >= 30), COUNTROWS(fact_loan))", "Lending", "Share of loans 30+ days past due. A lower rate is better."),
    ("fact_delinquency", "Delinquent Borrowers", "DISTINCTCOUNT(fact_delinquency[customerKey])", "Lending", "Distinct borrowers with a delinquency event."),
]
B_HIERS = {}

MODELS = [
    {"name": "PaymentsDepositsModel", "tables": A_TABLES, "rels": A_RELS, "measures": A_MEASURES, "hiers": A_HIERS},
    {"name": "LendingCreditRiskModel", "tables": B_TABLES, "rels": B_RELS, "measures": B_MEASURES, "hiers": B_HIERS},
]

# Natural label column per dimension (applied only when the table is in a model).
ISDEFAULTLABEL = {"dim_customer": "name", "dim_merchant": "name", "dim_account": "accountId",
                  "dim_product": "name", "dim_branch": "name", "dim_date": "date"}


In [ ]:
from sempy_labs.tom import connect_semantic_model

# Column descriptions (front-loaded, <=200 chars). Applied only to tables present in each model.
COLDESC = {
    ("dim_customer", "customerKey"): "Master customer key from entity resolution; the join key across payments, lending and fraud. Use this, not source IDs.",
    ("dim_customer", "email"): "Customer email; the identity attribute used to unify payments and lending records.",
    ("dim_customer", "name"): "Customer full name. Default label for the customer.",
    ("dim_customer", "segment"): "Customer segment (e.g., Retail, Wealth, SMB).",
    ("dim_customer", "region"): "Customer region for geographic analysis.",
    ("dim_customer", "homeCountry"): "Customer home country; compared against transaction geo for cross-border fraud.",
    ("dim_customer", "riskScore"): "Model risk score 0-1; higher is riskier. Average, do not sum.",
    ("dim_customer", "paymentsCustomerId"): "Source ID in the payments system (P-*). Lineage only; do not join facts on this.",
    ("dim_customer", "borrowerId"): "Source ID in the lending system (B-*). Lineage only; do not join facts on this.",
    ("dim_customer", "hasDeposits"): "True if the customer holds a deposit account (present in payments).",
    ("dim_customer", "hasLoans"): "True if the customer is a borrower (present in lending).",
    ("dim_merchant", "merchantId"): "Merchant key; join to transactions.",
    ("dim_merchant", "name"): "Merchant name. Default label for the merchant.",
    ("dim_merchant", "category"): "Merchant category description.",
    ("dim_merchant", "mcc"): "Merchant Category Code (numeric). Not additive; do not sum.",
    ("dim_merchant", "country"): "Merchant country.",
    ("dim_merchant", "riskFlag"): "1 if the merchant is flagged high-risk, else 0. Not additive.",
    ("dim_account", "accountId"): "Account key; join to transactions. Default label for the account.",
    ("dim_account", "customerId"): "Payments source customer ID (lineage). Use dim_customer[customerKey] for joins.",
    ("dim_account", "productId"): "Product held on the account; join to Product.",
    ("dim_account", "branchId"): "Owning branch; join to Branch.",
    ("dim_account", "balance"): "Current account balance in USD. Summed by Deposit Balance.",
    ("dim_account", "status"): "Account status (e.g., Open, Closed).",
    ("dim_account", "openedDate"): "Date the account was opened.",
    ("dim_product", "productId"): "Product key; join to accounts.",
    ("dim_product", "name"): "Product name. Default label for the product.",
    ("dim_product", "family"): "Product family grouping (top of the product hierarchy).",
    ("dim_branch", "branchId"): "Branch key; join to accounts.",
    ("dim_branch", "name"): "Branch name. Default label for the branch.",
    ("dim_branch", "region"): "Branch region (top of the geography hierarchy).",
    ("dim_date", "date"): "Calendar date. Default label for the date.",
    ("dim_date", "year"): "Calendar year. Not additive; do not sum.",
    ("dim_date", "quarter"): "Quarter number 1-4. Not additive.",
    ("dim_date", "month"): "Month number 1-12. Not additive.",
    ("dim_date", "day"): "Day of month. Not additive.",
    ("dim_date", "dateKey"): "Integer yyyymmdd surrogate key; join facts on this.",
    ("fact_transaction", "txnId"): "Transaction identifier. Grain of this table.",
    ("fact_transaction", "cardId"): "Card used for the transaction.",
    ("fact_transaction", "accountId"): "Account funding the transaction; join to Account.",
    ("fact_transaction", "customerId"): "Payments source customer ID (lineage); use customerKey for joins.",
    ("fact_transaction", "merchantId"): "Merchant of the transaction; join to Merchant.",
    ("fact_transaction", "amount"): "Transaction amount in USD. Summed by Transaction Amount.",
    ("fact_transaction", "currency"): "ISO currency code of the transaction.",
    ("fact_transaction", "timestamp"): "Transaction date-time.",
    ("fact_transaction", "channel"): "Channel (e.g., POS, Online, ATM).",
    ("fact_transaction", "geoCountry"): "Country where the transaction occurred; compared to home country for cross-border fraud.",
    ("fact_transaction", "mcc"): "Merchant Category Code (numeric). Not additive.",
    ("fact_transaction", "isFraud"): "True if the transaction is confirmed fraud. Drives fraud measures.",
    ("fact_transaction", "fraudType"): "Fraud pattern label when isFraud is true (e.g., cross-border, velocity).",
    ("fact_transaction", "customerKey"): "Master customer key; the join to Customer.",
    ("fact_transaction", "dateKey"): "Integer yyyymmdd; join to Date.",
    ("fact_fraud_alert", "alertId"): "Fraud alert identifier. Grain of this table.",
    ("fact_fraud_alert", "txnId"): "Transaction that triggered the alert.",
    ("fact_fraud_alert", "cardId"): "Card involved in the alert.",
    ("fact_fraud_alert", "customerId"): "Payments source customer ID (lineage).",
    ("fact_fraud_alert", "type"): "Alert type/category.",
    ("fact_fraud_alert", "severity"): "Alert severity (e.g., Low, Medium, High).",
    ("fact_fraud_alert", "status"): "Alert status; 'Open' counts toward Open Fraud Alerts.",
    ("fact_fraud_alert", "raisedAt"): "Date-time the alert was raised.",
    ("fact_fraud_alert", "customerKey"): "Master customer key; the join to Customer.",
    ("fact_loan", "loanId"): "Loan identifier. Grain of this table.",
    ("fact_loan", "borrowerId"): "Lending source borrower ID (lineage); use customerKey for joins.",
    ("fact_loan", "applicationId"): "Originating loan application.",
    ("fact_loan", "principal"): "Original loan principal in USD.",
    ("fact_loan", "rate"): "Annual interest rate percent. Not additive; average, do not sum.",
    ("fact_loan", "termMonths"): "Loan term in months. Not additive.",
    ("fact_loan", "balance"): "Outstanding loan balance in USD. Summed by Loan Book Balance.",
    ("fact_loan", "dpd"): "Days past due; 30+ counts as delinquent.",
    ("fact_loan", "originationDate"): "Date the loan was originated.",
    ("fact_loan", "customerKey"): "Master customer key; the join to Customer.",
    ("fact_delinquency", "loanId"): "Loan the delinquency event belongs to.",
    ("fact_delinquency", "eventId"): "Delinquency event identifier. Grain of this table.",
    ("fact_delinquency", "dpdBucket"): "Days-past-due bucket (e.g., 30, 60, 90).",
    ("fact_delinquency", "flaggedAt"): "Date-time the delinquency was flagged.",
    ("fact_delinquency", "borrowerId"): "Lending source borrower ID (lineage).",
    ("fact_delinquency", "customerKey"): "Master customer key; the join to Customer.",
}

# Build + enrich each model.
for m in MODELS:
    name = m["name"]
    tset = set(m["tables"])
    tables_map = {t: f"{SCHEMA}.{t}" for t in m["tables"]}
    labs.directlake.generate_direct_lake_semantic_model(
        dataset=name, tables=tables_map, source=LAKEHOUSE,
        source_type="Lakehouse", overwrite=True, refresh=False)
    with connect_semantic_model(dataset=name, readonly=False) as tom:
        for ft, fc, tt, tc in m["rels"]:
            try:
                tom.add_relationship(from_table=ft, from_column=fc, to_table=tt, to_column=tc,
                                     from_cardinality="Many", to_cardinality="One")
            except Exception as e:
                print("rel skip", name, ft, fc, ":", e)
        for t, nm, dax, folder, desc in m["measures"]:
            try:
                tom.add_measure(table_name=t, measure_name=nm, expression=dax,
                                display_folder=folder, description=desc)
            except Exception as e:
                print("measure skip", name, nm, ":", e)
        for (t, c), d in COLDESC.items():
            if t in tset:
                try:
                    tom.model.Tables[t].Columns[c].Description = d
                except Exception:
                    pass
                try:
                    tom.set_summarize_by(table_name=t, column_name=c, value="None")
                except Exception:
                    pass
        for t, c in ISDEFAULTLABEL.items():
            if t in tset:
                try:
                    tom.model.Tables[t].Columns[c].IsDefaultLabel = True
                except Exception:
                    pass
        for t, (hn, cols) in m["hiers"].items():
            try:
                tom.add_hierarchy(table_name=t, hierarchy_name=hn, columns=cols)
            except Exception as e:
                print("hierarchy skip", name, hn, ":", e)
    print("Built + enriched:", name)


In [ ]:
# Table-level descriptions (front-loaded; the data agent / Copilot uses these to pick the right table).
TABLEDESC = {
    "dim_customer": "Entity-resolved customers unifying payments and lending via customerKey. One row per person; the shared Customer dimension.",
    "dim_account": "Deposit accounts (checking/savings). One row per account; carries balance and links to product and branch.",
    "dim_merchant": "Merchants where cards are used. One row per merchant; carries category, MCC, and high-risk flag.",
    "dim_product": "Banking products (account types). One row per product; top of the product hierarchy.",
    "dim_branch": "Bank branches. One row per branch; top of the geography hierarchy.",
    "dim_date": "Calendar date dimension for time-based analysis. One row per day; join facts on dateKey.",
    "fact_transaction": "One row per card transaction; the grain for transaction volumes and fraud analysis.",
    "fact_fraud_alert": "Fraud alerts raised on transactions. One row per alert; Open status drives the alert measures.",
    "fact_loan": "Funded loans. One row per loan; carries principal, rate, balance, and days-past-due (credit exposure).",
    "fact_delinquency": "Loan delinquency events at 30/60/90 days past due. One row per event.",
}

from sempy_labs.tom import connect_semantic_model
for m in MODELS:
    with connect_semantic_model(dataset=m["name"], readonly=False) as tom:
        for t in m["tables"]:
            d = TABLEDESC.get(t)
            if d:
                try:
                    tom.model.Tables[t].Description = d
                except Exception as e:
                    print("table desc skip", m["name"], t, ":", e)
    print("Table descriptions set:", m["name"])


In [ ]:
# Direct Lake needs a full refresh to build the relationship indexes added above,
# otherwise cross-table measures fail with "relationship ... needs to be recalculated".
for m in MODELS:
    labs.refresh_semantic_model(dataset=m["name"], refresh_type="full")
    print("Refreshed (relationships rebuilt):", m["name"])


## (Optional) Auto-generate synonyms — NLTK WordNet + humanized field names

A **code-only** alternative to the Power BI Copilot "Suggest synonyms" UI (which has no REST/notebook API). For every table and column this:
1. **Humanizes** the name (`homeCountry` → *"home country"*, `dim_customer` → *"customer"*) as a base synonym, and
2. adds **WordNet** thesaurus synonyms for each real word token,

then writes them with `tom.set_synonym(culture="en-US", object=..., synonym_name=...)`.

> WordNet is a **general-English** thesaurus, so it's broad and a bit noisy on domain terms (IDs, codes, acronyms) — short/ambiguous tokens are skipped. It's a solid *baseline*; for domain-aware quality, generate synonyms with an LLM instead. These synonyms power **Power BI Q&A / Copilot over the model** (they do **not** flow to the ontology's own semantic enrichment).


In [ ]:
%pip install nltk


In [ ]:
import re, nltk
from sempy_labs.tom import connect_semantic_model
import sempy_labs as labs

nltk.download("wordnet", quiet=True)
from nltk.corpus import wordnet as wn

# tokens too short/ambiguous for a general thesaurus (IDs, codes) — humanize only, skip WordNet
STOP = {"id", "key", "dpd", "mcc", "txn", "card", "flag"}

def humanize(name: str) -> str:
    return re.sub(r"(?<=[a-z0-9])(?=[A-Z])", " ", name).replace("_", " ").lower()

def wn_synonyms(phrase: str, max_syn: int = 4) -> list:
    out = set()
    for token in phrase.split():
        if len(token) < 4 or token in STOP:
            continue
        for syn in wn.synsets(token):
            for lemma in syn.lemmas():
                w = lemma.name().replace("_", " ").lower()
                if w != token and " " not in w and w.isalpha():
                    out.add(w)
    return sorted(out)[:max_syn]

def candidates(name: str) -> set:
    h = humanize(name)
    return {h, *wn_synonyms(h)}

for m in MODELS:
    added = skipped = 0
    with connect_semantic_model(dataset=m["name"], readonly=False) as tom:
        tom.add_translation("en-US")   # ensure the en-US culture / linguistic schema exists

        # index existing synonyms by data binding so we only add NEW ones (idempotent re-runs)
        schema = tom.get_linguistic_schema("en-US") or {}
        existing = {}
        for ev in (schema.get("Entities") or {}).values():
            b = ev.get("Definition", {}).get("Binding", {})
            key = (b.get("ConceptualEntity"), b.get("ConceptualProperty"))
            for term in ev.get("Terms", []):
                existing.setdefault(key, set()).update(k.lower() for k in term)

        # build (object, table, column, candidate terms) targets
        targets = []
        for t in m["tables"]:
            base = re.sub(r"^(dim|fact)_", "", t)                 # dim_customer -> customer
            targets.append((tom.model.Tables[t], t, None, candidates(base)))
            for c in tom.model.Tables[t].Columns:
                if str(c.Type) != "RowNumber":
                    targets.append((c, t, c.Name, candidates(c.Name)))

        for obj, table, column, terms in targets:
            have = existing.setdefault((table, column), set())
            for s in terms:
                if not s or s == (column or table).lower():
                    continue
                if s.lower() in have:                              # already present -> skip
                    skipped += 1
                    continue
                try:
                    tom.set_synonym(culture="en-US", object=obj, synonym_name=s)
                    have.add(s.lower()); added += 1
                except Exception:
                    pass
    print(f"{m['name']}: added {added} new synonyms, skipped {skipped} already present")

# review what landed
display(labs.list_synonyms(dataset="PaymentsDepositsModel"))


In [ ]:
# ---------------- VERIFY (per model) ----------------
import sempy.fabric as fabric

print("=== PaymentsDepositsModel ===")
a = fabric.evaluate_dax(dataset="PaymentsDepositsModel", dax_string=
    'EVALUATE ROW("Txns",[Transaction Count], "TxnAmount",[Transaction Amount], '
    '"FraudLossRate",[Fraud Loss Rate], "DepositBal",[Deposit Balance])')
print(a.to_string(index=False))

print("\n=== LendingCreditRiskModel ===")
b = fabric.evaluate_dax(dataset="LendingCreditRiskModel", dax_string=
    'EVALUATE ROW("LoanBook",[Loan Book Balance], "Loans",[Loan Count], '
    '"DelinqRate",[Delinquency Rate], "DelinqBorrowers",[Delinquent Borrowers])')
print(b.to_string(index=False))

print("\nNote: the cross-domain question ('delinquent borrowers who also had fraud alerts' = 13) "
      "spans BOTH models, so no single semantic model answers it. That is the ontology's job (Module 3-4).")
